# Ceftriaxone — Phase-1 Site-Mask Diagnostic

**Is the site/instrument mask (top-500) weak because the site RF's default `max_features="sqrt"` dilutes importance?**

Sweeps the pooled site RF over 4 configs and reports, per config:
- OOB accuracy (site classification, 4 classes)
- importance concentration at top-{100, 500, 1000}
- masking-drop: `OOB_full − OOB_masked` with the top-500 bins zeroed

Reuses the exact 06-03d data pipeline (same split, downsampling, preprocessing, SEED).

In [ ]:
# ── CONFIG ──
MASK_TOP_K = 500
K_LIST = [100, 500, 1000]
RUN_NAME = "06-03d-Phase1-SiteMask-Diagnostic"
TARGET_RUN = "01-Run"


In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)


In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06b-Ceftriaxone-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")


In [ ]:
# ── Load Ceftriaxone — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")


In [ ]:
# ── Per-site species-stratified 90/10 split ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=0.10, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))}")


In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)

def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train")


In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
print("Per-site preprocessing done.")


In [ ]:
# ── Site RF config sweep + concentration + masking-drop vs K ──
CONFIGS = {
    "baseline": dict(n_estimators=300, min_samples_leaf=5),                  # max_depth=None, max_features='sqrt' (current 06-03d)
    "mf_0.2":   dict(n_estimators=300, min_samples_leaf=5, max_features=0.2),
    "mf_0.5":   dict(n_estimators=500, min_samples_leaf=5, max_features=0.5),
}
K_LIST = [100, 500, 1000]                 # importance-concentration checkpoints
DROP_K_LIST = [500, 1000, 1500, 2000]     # how many top bins to zero for the masking-drop

X_all = np.concatenate([client_train_pp[s][0] for s in SITE_ORDER])
y_site = np.concatenate([np.full(len(client_train_pp[s][0]), i) for i, s in enumerate(SITE_ORDER)])
print(f"Pooled site train: {len(X_all)} samples, 4 sites")

rows = []
for cfg_name, cfg in CONFIGS.items():
    print(f"\n=== {cfg_name}: {cfg} ===")
    rf = RandomForestClassifier(**cfg, n_jobs=-1, oob_score=True, random_state=SEED)
    rf.fit(X_all, y_site)
    imp = rf.feature_importances_
    oob_full = rf.oob_score_
    order = np.argsort(imp)[::-1]                      # most important first
    conc = {f"top{Kc}_conc": imp[order[:Kc]].sum() for Kc in K_LIST}
    print(f"  OOB={oob_full:.4f}, " + ", ".join(f"top{Kc}={conc[f'top{Kc}_conc']:.1%}" for Kc in K_LIST))

    for K in DROP_K_LIST:
        top_k = np.sort(order[:K])
        X_masked = X_all.copy(); X_masked[:, top_k] = 0.0
        rf_masked = RandomForestClassifier(**cfg, n_jobs=-1, oob_score=True, random_state=SEED)
        rf_masked.fit(X_masked, y_site)
        drop = oob_full - rf_masked.oob_score_
        rows.append({"config": cfg_name, "K": K, "oob_full": oob_full,
                     "oob_masked": rf_masked.oob_score_, "oob_drop": drop, **conc})
        print(f"    K={K}: masked OOB={rf_masked.oob_score_:.4f}, drop={drop:+.4f}")

df_diag = pd.DataFrame(rows)
df_diag.to_csv(OUT_DIR / "site_mask_drop_vs_k.csv", index=False)

print("\n===== Masking-drop (OOB_full - OOB_masked) vs K =====")
print(df_diag.pivot(index="config", columns="K", values="oob_drop").to_string())

fig, ax = plt.subplots(figsize=(8, 5))
for cfg_name in CONFIGS:
    sub = df_diag[df_diag["config"] == cfg_name]
    ax.plot(sub["K"], sub["oob_drop"], marker='o', label=cfg_name)
ax.set_xlabel("Top-K bins zeroed"); ax.set_ylabel("Site OOB drop")
ax.set_title(f"{DRUG_NAME} — site-mask drop vs K")
ax.legend(); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout(); plt.savefig(OUT_DIR / "site_mask_drop_vs_k.pdf", bbox_inches="tight"); plt.show()


---
**Done.** See `site_mask_diagnostic.csv` and `site_mask_concentration.pdf` in `results/`.

Interpretation:
- Higher `top-K_conc` = more concentrated site importance.
- Larger `oob_drop` = the top-500 bins carry more unique site/instrument signal.
- If `drop` stays ~0 (like species, ~1-2%), site signal is redundant → site masking limited.
- If `drop` is notably larger, instrument signal is localized → site masking is viable.
